[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/05_Foundations_in_Action.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/05_Foundations_in_Action.ipynb)

# Foundations in Action — One Dataset, All Four Concepts

The first four notebooks each taught one step of the workflow every data scientist follows when given a new dataset:

| Notebook | Concept | The question it answers |
|---|---|---|
| 01 | Statistics in learning | What does the data look like, and what patterns is a model actually learning? |
| 02 | Linear vs complex relationships | Are relationships smooth and proportional, or do they jump at thresholds? |
| 03 | Choosing your model | Based on what the data shows, which type of model should you start with? |
| 04 | Feature types and contributions | What type is each feature, and how does it contribute to one specific prediction? |

This notebook applies all four steps to one new dataset — **health insurance premiums** — showing that these are not separate topics. They are four sequential steps in every data analysis.

---

**The scenario**: You join an insurance company as a data scientist. You are given a dataset of 400 policyholders and asked to predict each person's annual premium. Walk through all four steps.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)
print('Ready.')


In [ ]:
n = 400

# ── Continuous ─────────────────────────────────────────────────────────────────
age        = np.random.uniform(25, 65, n)
bmi        = np.random.uniform(18, 40, n)
num_visits = np.random.poisson(3, n).clip(0, 10).astype(float)

# ── Ordinal ────────────────────────────────────────────────────────────────────
activity_level = np.random.choice([1, 2, 3, 4], n, p=[0.25, 0.35, 0.25, 0.15])
#   1=Sedentary  2=Light  3=Moderate  4=Active

# ── Binary ─────────────────────────────────────────────────────────────────────
smoker      = np.random.choice([0, 1], n, p=[0.75, 0.25])
has_chronic = np.random.choice([0, 1], n, p=[0.70, 0.30])

# ── Categorical → one-hot (north is baseline: all dummies = 0) ─────────────────
region       = np.random.choice(['north', 'south', 'east', 'west'], n, p=[0.30, 0.30, 0.20, 0.20])
region_south = (region == 'south').astype(float)
region_east  = (region == 'east').astype(float)
region_west  = (region == 'west').astype(float)

TRUE_INTERCEPT = 3000.0
TRUE_W = {
    'age':             50.0,    # +$50 per year of age
    'bmi':             30.0,    # +$30 per BMI unit
    'num_visits':      80.0,    # +$80 per annual doctor visit
    'activity_level': -200.0,   # -$200 per activity tier
    'smoker':        3500.0,    # smokers pay +$3500
    'has_chronic':   2500.0,    # chronic condition adds +$2500
    'region_south':   400.0,    # south vs north: +$400
    'region_east':    200.0,    # east vs north:  +$200
    'region_west':   -300.0,    # west vs north:  -$300
}
TRUE_INTERACTION = 1500.0  # extra cost when BOTH smoker=1 AND has_chronic=1

annual_premium = (
    TRUE_INTERCEPT
    + TRUE_W['age']            * age
    + TRUE_W['bmi']            * bmi
    + TRUE_W['num_visits']     * num_visits
    + TRUE_W['activity_level'] * activity_level
    + TRUE_W['smoker']         * smoker
    + TRUE_W['has_chronic']    * has_chronic
    + TRUE_W['region_south']   * region_south
    + TRUE_W['region_east']    * region_east
    + TRUE_W['region_west']    * region_west
    + TRUE_INTERACTION         * smoker * has_chronic   # joint risk
    + np.random.normal(0, 300, n)
)

FEATURES = ['age', 'bmi', 'num_visits', 'activity_level',
            'smoker', 'has_chronic',
            'region_south', 'region_east', 'region_west']

df = pd.DataFrame({
    'age': age, 'bmi': bmi, 'num_visits': num_visits,
    'activity_level': activity_level,
    'smoker': smoker, 'has_chronic': has_chronic,
    'region': region,
    'region_south': region_south, 'region_east': region_east, 'region_west': region_west,
    'annual_premium': annual_premium
})

print(f'Dataset: {n} policyholders, {len(FEATURES)} features spanning all four types')
print(f'Annual premium range: ${annual_premium.min():,.0f}  to  ${annual_premium.max():,.0f}')
print(f'Smokers: {smoker.mean():.0%}   Has chronic condition: {has_chronic.mean():.0%}')
print(f'Both smoker + chronic (interaction group): {(smoker * has_chronic).mean():.0%}')
df[['age', 'bmi', 'num_visits', 'activity_level', 'smoker',
    'has_chronic', 'region', 'annual_premium']].head(6)


## Part 1 — Distributions and Correlation (Notebook 01 Concepts)

The first question to ask about any dataset: **what does the data look like?**

Notebook 01 established that a model learns by adjusting weights until the weighted sum of features matches the target. Before training, use statistics and plots to see those patterns yourself.

Three things to look at:

1. **Target distribution** — unimodal, bimodal, or skewed? A bimodal shape signals that one feature sharply splits the population into two cost groups.
2. **Feature distributions** — what values will the model learn from? Rare values, outliers, imbalanced classes?
3. **Correlation with target** — which features carry signal? Use the right measure for each type (Pearson, Spearman, Point-biserial).


In [ ]:
# ── Feature distributions ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(14, 10))

feat_info = [
    ('age',            'Continuous',         '#1565C0'),
    ('bmi',            'Continuous',         '#1565C0'),
    ('num_visits',     'Discrete',           '#7E57C2'),
    ('activity_level', 'Ordinal',            '#26A69A'),
    ('smoker',         'Binary',             '#FF7043'),
    ('has_chronic',    'Binary',             '#FF7043'),
    ('region_south',   'Categorical dummy',  '#F9A825'),
    ('region_east',    'Categorical dummy',  '#F9A825'),
    ('region_west',    'Categorical dummy',  '#F9A825'),
]

for ax, (feat, ftype, color) in zip(axes.flat, feat_info):
    if ftype in ('Binary', 'Categorical dummy', 'Ordinal'):
        counts = df[feat].value_counts().sort_index()
        ax.bar([str(v) for v in counts.index], counts.values, color=color, edgecolor='white')
    elif ftype == 'Discrete':
        mx = int(df[feat].max()) + 2
        ax.hist(df[feat], bins=[v - 0.5 for v in range(mx)],
                color=color, edgecolor='white')
    else:
        ax.hist(df[feat], bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{feat}  [{ftype}]', fontsize=9)

plt.suptitle('Step 1 — Feature distributions: what values will the model learn from?',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

# ── Target distribution ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(df['annual_premium'], bins=45, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Annual premium ($)')
ax.set_title('Target distribution — annual_premium', fontsize=11)
plt.tight_layout()
plt.show()

print('The target distribution shows two peaks (bimodal).')
print('This means one feature divides policyholders into two distinct cost groups.')
print('Identifying which feature drives this split is what correlation analysis does.')


In [ ]:
results = {}

for feat in ['age', 'bmi']:
    r, _ = stats.pearsonr(df[feat], df['annual_premium'])
    results[feat] = ('Pearson r', round(r, 3))

for feat in ['num_visits', 'activity_level']:
    r, _ = stats.spearmanr(df[feat], df['annual_premium'])
    results[feat] = ('Spearman rho', round(r, 3))

for feat in ['smoker', 'has_chronic']:
    r, _ = stats.pointbiserialr(df[feat], df['annual_premium'])
    results[feat] = ('Point-biserial', round(r, 3))

for feat in ['region_south', 'region_east', 'region_west']:
    r, _ = stats.pearsonr(df[feat], df['annual_premium'])
    results[feat] = ('Pearson r (dummy)', round(r, 3))

corr_df = (pd.DataFrame([{'feature': f, 'measure': m, 'r': v} for f, (m, v) in results.items()])
             .sort_values('r'))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4CAF50' if v > 0 else '#e57373' for v in corr_df['r']]
bars = ax.barh(corr_df['feature'], corr_df['r'], color=colors)
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Correlation with annual_premium')
ax.set_title('Which features carry signal? — appropriate measure applied per type',
             fontsize=11)

for bar, row in zip(bars, corr_df.itertuples()):
    x = row.r
    ha = 'left' if x >= 0 else 'right'
    ax.text(x + 0.01 if x >= 0 else x - 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{row.measure}: {x:.3f}',
            va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.show()

print('smoker:         highest — this is the feature creating the bimodal distribution')
print('has_chronic:    second — also a strong predictor')
print('activity_level: negative — more active people pay lower premiums')
print('region_*:       weak — small regional adjustments, not a primary driver')


## Part 2 — Are Relationships Linear? (Notebook 02 Concepts)

Correlation tells you *how strongly* features move with the target. Plots tell you *how* they move.

Notebook 02 introduced three shapes to look for:

- **Linear**: target changes smoothly and proportionally — linear regression captures this with a coefficient
- **Threshold**: target jumps sharply at a specific value — a decision tree learns this naturally as a split
- **Interaction**: the effect of one feature depends on another — needs an interaction term or a tree

**Why this matters**: if your data has threshold effects or interactions, a linear model fits a smooth line through a pattern that requires a sharp boundary — and it will systematically mispredict one group of observations.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# ── Row 1: continuous features ────────────────────────────────────────────────
for ax, feat in zip(axes[0], ['age', 'bmi', 'num_visits']):
    ax.scatter(df[feat], df['annual_premium'], alpha=0.25, s=12, color='steelblue')
    m, b = np.polyfit(df[feat], df['annual_premium'], 1)
    xs = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(xs, m * xs + b, color='#e57373', lw=2.5, label='linear fit')
    ax.set_xlabel(feat)
    ax.set_ylabel('annual premium ($)')
    ax.set_title(f'{feat} — linear?', fontsize=10)
    ax.legend(fontsize=8)

# ── Row 2: binary and ordinal ─────────────────────────────────────────────────
for ax, feat in zip(axes[1, :2], ['smoker', 'has_chronic']):
    groups = [df[df[feat] == v]['annual_premium'].values for v in [0, 1]]
    bp = ax.boxplot(groups, labels=['No (0)', 'Yes (1)'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#4CAF50')
    bp['boxes'][1].set_facecolor('#e57373')
    ax.set_xlabel(feat)
    ax.set_ylabel('annual premium ($)')
    ax.set_title(f'{feat} — threshold jump?', fontsize=10)

mean_by_act = df.groupby('activity_level')['annual_premium'].mean()
axes[1, 2].bar(mean_by_act.index.astype(str), mean_by_act.values,
               color='#26A69A', edgecolor='white')
axes[1, 2].set_xlabel('activity_level (1=Sedentary, 4=Active)')
axes[1, 2].set_ylabel('Mean annual premium ($)')
axes[1, 2].set_title('activity_level — smooth step-down?', fontsize=10)

plt.suptitle('Step 2 — Relationship shapes: linear, threshold, or interaction?',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ── Interaction check: smoker x has_chronic ────────────────────────────────────
group_labels = {(0, 0): 'Neither', (1, 0): 'Smoker only',
                (0, 1): 'Chronic only', (1, 1): 'Both'}
means = {}
for (s, c), label in group_labels.items():
    mask = (df['smoker'] == s) & (df['has_chronic'] == c)
    means[label] = df[mask]['annual_premium'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
colors_g = ['#4CAF50', '#FFA726', '#42A5F5', '#e57373']
bars = ax.bar(list(means.keys()), list(means.values()), color=colors_g, edgecolor='white')
for bar, (label, v) in zip(bars, means.items()):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 80,
            f'${v:,.0f}', ha='center', fontsize=9)
ax.set_ylabel('Mean annual premium ($)')
ax.set_title('Interaction check: smoker x has_chronic — does having BOTH cost more than the sum?',
             fontsize=10)
plt.tight_layout()
plt.show()

additive_expected = (means['Neither']
                     + (means['Smoker only'] - means['Neither'])
                     + (means['Chronic only'] - means['Neither']))
print(f'If effects were purely additive, expected premium for Both:  ${additive_expected:,.0f}')
print(f'Actual mean premium for Both (smoker + chronic):             ${means["Both"]:,.0f}')
print(f'Interaction surplus:                                          ${means["Both"] - additive_expected:,.0f}')
print()
print('The "Both" group costs more than the sum of the two individual effects.')
print('This interaction cannot be captured by a linear model without an engineered feature.')
print('A decision tree finds it automatically: IF smoker=1, THEN check has_chronic.')


## Part 3 — Choosing the Right Model (Notebook 03 Concepts)

Applying the checklist from Notebook 03:

| Question | Answer for this dataset |
|---|---|
| Continuous target? | Yes — regression problem |
| Features correlate with target? | Yes — smoker, has_chronic, activity_level are strong |
| Relationships linear for all features? | age, bmi, num_visits: yes. smoker, has_chronic: **threshold jump** |
| Interaction effects? | **Yes** — smoker × has_chronic costs more than either alone |

**Prediction**: Linear Regression will handle the continuous features well but will struggle with the interaction. The Decision Tree should outperform it by learning the split: "IF smoker AND has_chronic, add extra cost."

We hold out 20% of the data to measure how well each model predicts observations it has not seen during training.


In [ ]:
idx_all = np.arange(len(df))
idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=42)

X = df[FEATURES].values
y = df['annual_premium'].values

X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr   = LinearRegression().fit(X_train_sc, y_train)
tree = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_train, y_train)

lr_pred   = lr.predict(X_test_sc)
tree_pred = tree.predict(X_test)

lr_mae, lr_r2     = mean_absolute_error(y_test, lr_pred),   r2_score(y_test, lr_pred)
tree_mae, tree_r2 = mean_absolute_error(y_test, tree_pred), r2_score(y_test, tree_pred)

# ── Actual vs predicted ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, name, mae, r2 in zip(
    axes,
    [lr_pred, tree_pred],
    ['Linear Regression', 'Decision Tree (max_depth=5)'],
    [lr_mae, tree_mae],
    [lr_r2,  tree_r2]
):
    ax.scatter(y_test, pred, alpha=0.45, s=14, color='steelblue')
    lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='perfect prediction')
    ax.set_xlabel('Actual premium ($)')
    ax.set_ylabel('Predicted premium ($)')
    ax.set_title(f'{name}  |  MAE = ${mae:,.0f}   R² = {r2:.3f}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Step 3 — Model comparison on held-out test data', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ── Residuals colored by interaction group ─────────────────────────────────────
smoker_test  = df['smoker'].values[idx_test]
chronic_test = df['has_chronic'].values[idx_test]
both_test    = (smoker_test == 1) & (chronic_test == 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, pred, name in zip(axes, [lr_pred, tree_pred],
                           ['Linear Regression', 'Decision Tree']):
    residuals = y_test - pred
    point_colors = ['#e57373' if b else '#4CAF50' for b in both_test]
    ax.scatter(pred, residuals, c=point_colors, alpha=0.5, s=14)
    ax.axhline(0, color='black', lw=1.2)
    ax.set_xlabel('Predicted premium ($)')
    ax.set_ylabel('Residual (actual − predicted)')
    ax.set_title(f'{name} residuals  |  Red = both smoker + chronic', fontsize=10)

axes[0].legend(handles=[
    Patch(facecolor='#e57373', label='Smoker + chronic (interaction group)'),
    Patch(facecolor='#4CAF50', label='All others')], fontsize=8)

plt.suptitle('Step 3 — Does the model systematically mispredict the interaction group?',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print(f'Linear Regression:        MAE = ${lr_mae:,.0f}   R² = {lr_r2:.3f}')
print(f'Decision Tree (depth=5):  MAE = ${tree_mae:,.0f}   R² = {tree_r2:.3f}')
print()
print('The red dots (interaction group) cluster below zero in the linear model residual plot:')
print('it consistently under-predicts the most expensive policyholders — those who are')
print('BOTH smokers AND have a chronic condition.')
print('The decision tree captures this with a split: IF smoker=1, THEN check has_chronic.')


## Part 4 — Feature Types and Contributions (Notebook 04 Concepts)

Notebook 04 showed that every feature — regardless of its type — contributes through the same arithmetic:

```
prediction = intercept + weight_1 × value_1 + weight_2 × value_2 + … + weight_n × value_n
```

The **type** of the feature determines what values it can take, and therefore when it speaks and when it is silent:

| Feature type | When silent (contribution = 0) | When active |
|---|---|---|
| Continuous | Only if value = 0 (rare) | Always — contribution scales with value |
| Discrete | When count = 0 | Proportional to the count |
| Ordinal | Rarely (only if level = 0) | Higher level → larger magnitude |
| Binary | **When value = 0** — feature absent | When value = 1 — full weight passes through |
| Categorical (dummy) | **All inactive columns (value = 0)** | Only the one active column |

We now look at one specific policyholder and decompose the prediction feature by feature.


In [ ]:
# Recover raw-space weights from the scaled model
# raw_weight_j = scaled_weight_j / std_j
# adjusted intercept: absorbs the mean shifts from scaling
raw_weights        = lr.coef_ / scaler.scale_
adjusted_intercept = lr.intercept_ - np.dot(raw_weights, scaler.mean_)

# Pick an observation: non-smoker, has chronic condition, west region
#   → smoker=0 (SILENT), has_chronic=1 (ACTIVE), region_west=1 (ACTIVE)
mask = (df['smoker'] == 0) & (df['has_chronic'] == 1) & (df['region'] == 'west')
OBS  = df[mask].index[0]

obs      = df.iloc[OBS]
obs_vals = df[FEATURES].iloc[OBS].values
contrib  = raw_weights * obs_vals

ftype_map = {
    'age':            'Continuous',
    'bmi':            'Continuous',
    'num_visits':     'Discrete',
    'activity_level': 'Ordinal',
    'smoker':         'Binary',
    'has_chronic':    'Binary',
    'region_south':   'Categorical',
    'region_east':    'Categorical',
    'region_west':    'Categorical',
}

contrib_df = pd.DataFrame({
    'feature':      FEATURES,
    'ftype':        [ftype_map[f] for f in FEATURES],
    'value':        obs_vals,
    'weight':       raw_weights,
    'contribution': contrib,
})

predicted = adjusted_intercept + contrib.sum()

print(f'Observation {OBS}  |  region = {df["region"].iloc[OBS]}  |'
      f'  smoker = {int(obs["smoker"])}  |  has_chronic = {int(obs["has_chronic"])}')
print(f'Actual premium = ${obs["annual_premium"]:,.0f}    Model prediction = ${predicted:,.0f}')
print()
print(f'  {"Feature":<18} {"Type":<12} {"Value":>8} {"Weight":>9} {"Contribution":>14}')
print('  ' + '-' * 67)
for _, row in contrib_df.iterrows():
    silent = '  <- SILENT' if row['value'] == 0 else ''
    print(f'  {row["feature"]:<18} {row["ftype"]:<12} {row["value"]:>8.2f}'
          f' {row["weight"]:>9.1f} {row["contribution"]:>13.2f}{silent}')
print('  ' + '-' * 67)
print(f'  {"Intercept":<18} {" ":>12} {" ":>8} {" ":>9} {adjusted_intercept:>13.2f}')
print(f'  {"TOTAL":<18} {" ":>12} {" ":>8} {" ":>9} {predicted:>13.2f}')

# ── Contribution bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
bar_colors = []
for v, c in zip(obs_vals, contrib):
    if v == 0:
        bar_colors.append('#cccccc')
    elif c > 0:
        bar_colors.append('#4CAF50')
    else:
        bar_colors.append('#e57373')

bars = ax.barh(FEATURES, contrib, color=bar_colors, edgecolor='white', height=0.6)

for bar, row in zip(bars, contrib_df.itertuples()):
    v, w, c = row.value, row.weight, row.contribution
    y_mid = bar.get_y() + bar.get_height() / 2
    if v == 0:
        ax.text(50, y_mid, f'value=0  →  {w:.1f} × 0 = $0   SILENT',
                va='center', ha='left', fontsize=8, color='#888888', style='italic')
    else:
        xpos = c + 60 if c >= 0 else c - 60
        ha   = 'left' if c >= 0 else 'right'
        ax.text(xpos, y_mid, f'{w:.1f} × {v:.1f} = ${c:,.0f}',
                va='center', ha=ha, fontsize=8)

ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Contribution to predicted premium ($)')
ax.set_title(f'Step 4 — Feature contributions for observation {OBS}  '
             f'(non-smoker, chronic, west region)', fontsize=11)
ax.legend(handles=[
    Patch(facecolor='#4CAF50', label='Positive — raises premium'),
    Patch(facecolor='#e57373', label='Negative — lowers premium'),
    Patch(facecolor='#cccccc', label='Silent — value is 0'),
], loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()


## Putting It Together — The Four-Step Workflow

The four notebooks are not separate topics. They are **four sequential steps** that every data analysis follows:

```
New dataset
    |
    v
Step 1  --  Distributions & correlation         (Notebook 01)
            What does the data look like?
            Which features carry signal?
    |
    v
Step 2  --  Relationship shapes                 (Notebook 02)
            Are features linearly related to the target?
            Threshold effects? Interactions?
    |
    v
Step 3  --  Model choice                        (Notebook 03)
            Linear / logistic regression for smooth patterns
            Decision tree when thresholds or interactions exist
    |
    v
Step 4  --  Feature types & contributions       (Notebook 04)
            What type is each feature?
            contribution = weight x value
            Binary / categorical: silent when value = 0
```

Applied to the insurance dataset:

| Step | Finding |
|---|---|
| 1 — Distributions | Bimodal target; smoker and has_chronic correlate most strongly |
| 2 — Shapes | age/bmi/num_visits are linear; smoker and has_chronic create threshold jumps; their joint effect is a $1500+ interaction |
| 3 — Model choice | Decision tree outperforms linear regression — it captures the smoker × chronic interaction via two splits |
| 4 — Contributions | Binary features are silent when 0; only the active region dummy speaks; per-feature contributions decompose any prediction |

**The connecting thread**: every concept comes back to `weight × value`. The type of a feature determines what values it can take — and therefore when it has a voice in a prediction and when it is silent.


## Key Takeaways

### Step 1 — Look before you model (Notebook 01)
- Plot the target distribution first — bimodal means one feature sharply splits the population
- Use the right correlation measure per type: Pearson, Spearman, Point-biserial, Pearson-per-dummy
- High correlation → feature carries signal; near zero → weak predictor

### Step 2 — Check relationship shapes (Notebook 02)
- Scatter plots for continuous: elongated cloud = linear; curved or banded = non-linear
- Box plots for binary: a sharp vertical gap between boxes = threshold effect
- Bar chart for ordinal/categorical: does the mean change smoothly across levels?
- Two-way group means reveal interactions: combine feature A and feature B and check if one depends on the other

### Step 3 — Choose the right model (Notebook 03)
- Smooth, proportional patterns → Linear or Logistic Regression
- Threshold effects or interactions → Decision Tree (or Random Forest)
- Always compare: fit both, plot residuals, and look for systematic errors in one subgroup

### Step 4 — Feature types and contributions (Notebook 04)
- Every feature contributes through `weight × value` — regardless of type
- Binary features: value = 0 → silent; value = 1 → full weight passes through
- Categorical dummies: only the one active column contributes; all others are silent
- Scale features before training so all types compete on equal footing

### Why none of the steps is optional
Skipping a step leads to a predictable failure:
- Skip step 1 → miss the bimodal target (wrong model family)
- Skip step 2 → use a linear model on non-linear data (systematic prediction errors)
- Skip step 3 → choose the wrong model (leave accuracy on the table)
- Skip step 4 → treat predictions as black boxes (can't explain or debug errors)
